# Purpose of Notebook
- Describes the usage of *"DetectOffice(Box, Office,VertPoint)"* function.

- General usage

    Use to find the locations of offices in a given page.

- Advice in using.
    
    Detection works better when the set of items to search is smaller. 
    
    Recommended to filter Box using layout information. 
    
    Step C of this notebook demonstrates this.
    
- Input/Output of function.

    Inputs are 
    1. Dictionary of text items with locational information, 
    2. Name of the office to detect, and 
    3. Height of the horizontal line used to vertically seperate a page in to two pieces.
    
    $\;\;\;\;\;\;$
    
    Output is a list of office with its location in a page.

In [1]:
import sys, json, os, cv2
import pandas as pd

In [2]:
Year,Showa='1950','25'
Quality='HQ'

In [82]:
df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

# Requirements
1. **Page to extract data from.**

    Demonstrated at Step A.
2. **Google Vision output**

    Demonstrated at Step B. Step C filters the output for improving precision.
3. **Name of Office**

    Demonstrated at Step A.

# A. Set Page 

In [83]:
Page=20
path='C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/'
file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+Quality+'/Page'+'{:03d}'.format(Page)+'.png'
img=cv2.imread(os.path.join(path,file_path))

cv2.namedWindow("Resized_Window", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Resized_Window", 1280, 1440)
cv2.imshow("Resized_Window",img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [84]:
BookPage=2*Page+16
Rows=df[(df['Page']==BookPage)]
NxRows=df[(df['Page']==BookPage+1)]
if len(Rows)==0:
    print('No Office at Right Side. Page'+str(BookPage))
if len(NxRows)==0:
    print('No Office at Left Side. Page'+str(BookPage+1))

# B. Scan image with Google Vision

In [85]:
sys.path.append(path+'Tokyo_Jobs/Data_Collection/Extract_Data')
from Read import Vision
from google.cloud import vision
from google.cloud.vision_v1 import types
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path+'Tokyo_Jobs\\Environment\\GoogleVision\\practice-302516-01e0f7245b03.json'
# Instantiates a client
client = vision.ImageAnnotatorClient()

In [86]:
texts=Vision(img,'zh',client)

# C. Filter items


In [87]:
sys.path.append(path+'Tokyo_Jobs\\Data_Collection\\Split_Page')
from Split import VertPart
sys.path.append(path+'Tokyo_Jobs\\Data_Collection\\General')
from PageCut import HoriPart
from Organize import Filter, ConvertDict
origin=path
file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/HQ/Page'+'{:03d}'.format(Page)+'.jpg'
path=os.path.join(origin,file_path)
path

'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/HQ/Page020.jpg'

In [88]:
HoriPoint=HoriPart(img,Page,texts)[0]

try:
    VertPoint=VertPart(path)[1]
    VertPoint=HH//3
except:
    print('Failed detecting Vertical Point')
    HH,WW=img.shape[:2]
    VertPoint=HH//3
    

Horizontal Line was not automatically detected... Defining line arbitrariry...


In [98]:
#Right Page
Page=Rows['Page'].tolist()[0]
BoxR=Filter(Page,texts,HoriPoint)
BoxR=ConvertDict(BoxR)

In [99]:
#Left Page
Page=NxRows['Page'].tolist()[0]
BoxL=Filter(Page,texts,HoriPoint)
BoxL=ConvertDict(BoxL)

# Main Code: Extract Office info
- Input
     1. List of offices included in the page.
     2. Dictionary of items.
     

- Output
     1. List of office locations

In [100]:
from Show import Show

In [101]:
from DetectOffice import DetectOffice
from Organize import FilterBox
OfficeList=Rows['Office'].tolist()

In [102]:
#RightPage
OfficeList=Rows['Office'].tolist()
LocList=[]
for Office in OfficeList:
    OfficeInfo=DetectOffice(BoxR, Office,VertPoint)
    if OfficeInfo==None:
        continue
    LocList.append(OfficeInfo)
    BoxR=FilterBox(BoxR,LocList,VertPoint)

In [103]:
Show(img,LocList)

In [104]:
#LeftPage
OfficeList=NxRows['Office'].tolist()
LocList=[]
for Office in OfficeList:
    OfficeInfo=DetectOffice(BoxL, Office,VertPoint)
    if OfficeInfo==None:
        continue
    LocList.append(OfficeInfo)
    BoxL=FilterBox(BoxL,LocList,VertPoint)

In [105]:
Show(img,LocList)

# Show in image

In [20]:
print(OfficeList)

['管理係', '指導係', '保護係', '援護係', '水上民生館', '淀橋寮', '戸山寮', '大原寮', '品川相談所', '上野相談所', '東京相談所']


In [21]:
[d['description'] for d in BoxL]

['川', '上傳', '助', '山', '寮', '三七']